In [ ]:
!pip install pandas numpy scikit-learn

!pip install scikit-learn

In [ ]:
 # STEP 1 import libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df=pd.read_csv("/content/IMDB Dataset 2.csv")
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [ ]:
df['review']=df['review'].str.lower()
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


In [ ]:
# removes html tags
def removetags(df,column):
  def cleanhtml(text):
    clean=re.compile(r'<.*?>')
    return re.sub(clean,'',text)

  df[column]=df[column].astype(str)
  df[column]=df[column].apply(cleanhtml)
  return df

  # OR
 #[
   # def removetags(df, column):
   # df[column] = df[column].astype(str)
   # df[column] = df[column].str.replace(r'<.*?>', '', regex=True)
   # return df
  #]

  removetags(df,'reviews')

In [ ]:
# removes punctuation
def rmpunct(df,column):
  df[column]=df[column].str.replace(r'[^\w\s]','',regex=True)
  return df

In [ ]:
rmpunct(df,'review')  #removes punctuation


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


In [ ]:
df["review"][2]

'i thought this was a wonderful way to spend time on a too hot summer weekend sitting in the air conditioned theater and watching a lighthearted comedy the plot is simplistic but the dialogue is witty and the characters are likable even the well bread suspected serial killer while some may be disappointed when they realize this is not match point 2 risk addiction i thought it was proof that woody allen is still fully in control of the style many of us have grown to lovebr br this was the most id laughed at one of woodys comedies in years dare i say a decade while ive never been impressed with scarlet johanson in this she managed to tone down her sexy image and jumped right into a average but spirited young womanbr br this may not be the crown jewel of his career but it was wittier than devil wears prada and more interesting than superman a great comedy to go see with friends'

In [ ]:
abbreviations = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek You (also a chat program)",
    "ILU": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "KISS": "Keep It Simple, Stupid",
    "LDR": "Long Distance Relationship",
    "LMAO": "Laugh My A.. Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA?": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laugher",
    "TFW": "That Feeling When",
    "MFW": "My Face When",
    "MRW": "My Reaction When",
    "IFYP": "I Feel Your Pain",
    "LOL": "Laughing Out Loud",
    "TNTL": "Trying Not To Laugh",
    "JK": "Just Kidding",
    "IDC": "I Don’t Care",
    "ILY": "I Love You",
    "IMU": "I Miss You",
    "ADIH": "Another Day In Hell",
    "ZZZ": "Sleeping, Bored, Tired",
    "WYWH": "Wish You Were Here",
    "TIME": "Tears In My Eyes",
    "BAE": "Before Anyone Else",
    "FIMH": "Forever In My Heart",
    "BSAAW": "Big Smile And A Wink",
    "BWL": "Bursting With Laughter",
    "BFF": "Best Friends Forever",
    "CSL": "Can’t Stop Laughing"
}


In [ ]:
def correctwor(df, column):
    newt = []

    for text in df[column]:
        neww = []
        for word in text.split():
            if word.upper() in abbreviations:
                neww.append(abbreviations[word.upper()])
            else:
                neww.append(word)

        newt.append(' '.join(neww))

    df[column] = newt
    return df


In [ ]:
correctwor(df,'review')

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend Te...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the Tears In My Eyes of...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


In [ ]:
!pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
def remove_sw(df,column):
  stop_words=set(stopwords.words('english'))
  df[column]=df[column].astype(str)
  df[column]=df[column].apply(
      lambda text: ' '.join(
          word for word in text.split()
          if word.lower()not in stop_words
      )
  )
  return df


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
remove_sw(df,'review')

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend Tears Eyes hot sum...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love Tears Eyes money visually ...,positive
...,...,...
49995,thought movie right good job wasnt creative or...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,catholic taught parochial elementary schools n...,negative
49998,im going disagree previous comment side maltin...,negative


In [ ]:
!pip install spacy


In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp=spacy.load('en_core_web_sm')

In [ ]:
def tokenn(df, column):
    df[column] = df[column].astype(str)

    def tokentext(text):
        doc = nlp(text)
        tokens = []
        for token in doc:
            tokens.append(token.text)
        return tokens

    newt = []
    for text in df[column]:
        newt.append(tokentext(text))

    df["Tokens"] = newt
    return df


In [ ]:
tokenn(df,'review')

,review,sentiment,Tokens
0,one reviewers mentioned watching 1 oz episode ...,positive,"[one, reviewers, mentioned, watching, 1, oz, e..."
1,wonderful little production br br filming tech...,positive,"[wonderful, little, production, br, br, filmin..."
2,thought wonderful way spend Tears Eyes hot sum...,positive,"[thought, wonderful, way, spend, Tears, Eyes, ..."
3,basically theres family little boy jake thinks...,negative,"[basically, there, s, family, little, boy, jak..."
4,petter matteis love Tears Eyes money visually ...,positive,"[petter, matteis, love, Tears, Eyes, money, vi..."
...,...,...,...
49995,thought movie right good job wasnt creative or...,positive,"[thought, movie, right, good, job, was, nt, cr..."
49996,bad plot bad dialogue bad acting idiotic direc...,negative,"[bad, plot, bad, dialogue, bad, acting, idioti..."
49997,catholic taught parochial elementary schools n...,negative,"[catholic, taught, parochial, elementary, scho..."
49998,im going disagree previous comment side maltin...,negative,"[i, m, going, disagree, previous, comment, sid..."


In [ ]:
df

,review,sentiment,Tokens
0,one reviewers mentioned watching 1 oz episode ...,positive,"[one, reviewers, mentioned, watching, 1, oz, e..."
1,wonderful little production br br filming tech...,positive,"[wonderful, little, production, br, br, filmin..."
2,thought wonderful way spend Tears Eyes hot sum...,positive,"[thought, wonderful, way, spend, Tears, Eyes, ..."
3,basically theres family little boy jake thinks...,negative,"[basically, there, s, family, little, boy, jak..."
4,petter matteis love Tears Eyes money visually ...,positive,"[petter, matteis, love, Tears, Eyes, money, vi..."
...,...,...,...
49995,thought movie right good job wasnt creative or...,positive,"[thought, movie, right, good, job, was, nt, cr..."
49996,bad plot bad dialogue bad acting idiotic direc...,negative,"[bad, plot, bad, dialogue, bad, acting, idioti..."
49997,catholic taught parochial elementary schools n...,negative,"[catholic, taught, parochial, elementary, scho..."
49998,im going disagree previous comment side maltin...,negative,"[i, m, going, disagree, previous, comment, sid..."


In [ ]:

df

,review,sentiment,Tokens
0,one reviewers mentioned watching 1 oz episode ...,positive,o n e r e v i e w e r s m e n t i o n e d ...
1,wonderful little production br br filming tech...,positive,w o n d e r f u l l i t t l e p r o d u c ...
2,thought wonderful way spend Tears Eyes hot sum...,positive,t h o u g h t w o n d e r f u l w a y s ...
3,basically theres family little boy jake thinks...,negative,b a s i c a l l y t h e r e s f a m i l ...
4,petter matteis love Tears Eyes money visually ...,positive,p e t t e r m a t t e i s l o v e T e a ...
...,...,...,...
49995,thought movie right good job wasnt creative or...,positive,t h o u g h t m o v i e r i g h t g o o ...
49996,bad plot bad dialogue bad acting idiotic direc...,negative,b a d p l o t b a d d i a l o g u e b ...
49997,catholic taught parochial elementary schools n...,negative,c a t h o l i c t a u g h t p a r o c h i ...
49998,im going disagree previous comment side maltin...,negative,i m g o i n g d i s a g r e e p r e v ...


In [ ]:
def corpus(df,column):
  single_corpus=[]
  for tokenl in df[column]:
    for token in tokenl:
      single_corpus.append(token)
  return single_corpus

In [ ]:
df

,review,sentiment,Tokens
0,one reviewers mentioned watching 1 oz episode ...,positive,o n e r e v i e w e r s m e n t i o n e d ...
1,wonderful little production br br filming tech...,positive,w o n d e r f u l l i t t l e p r o d u c ...
2,thought wonderful way spend Tears Eyes hot sum...,positive,t h o u g h t w o n d e r f u l w a y s ...
3,basically theres family little boy jake thinks...,negative,b a s i c a l l y t h e r e s f a m i l ...
4,petter matteis love Tears Eyes money visually ...,positive,p e t t e r m a t t e i s l o v e T e a ...
...,...,...,...
49995,thought movie right good job wasnt creative or...,positive,t h o u g h t m o v i e r i g h t g o o ...
49996,bad plot bad dialogue bad acting idiotic direc...,negative,b a d p l o t b a d d i a l o g u e b ...
49997,catholic taught parochial elementary schools n...,negative,c a t h o l i c t a u g h t p a r o c h i ...
49998,im going disagree previous comment side maltin...,negative,i m g o i n g d i s a g r e e p r e v ...


In [ ]:
whole_corpus=corpus(df,'Tokens')

In [ ]:
print(whole_corpus)

Buffered data was truncated after reaching the output size limit.

In [ ]:
corpuslength=len(whole_corpus)

In [ ]:
vocublary=set(whole_corpus)

In [ ]:
vocublarylength=len(vocublary)

In [ ]:
print(f'corpus is {corpuslength} ,vocabulary is {vocublarylength}')

corpus is 6294244 ,vocabulary is 180633


In [ ]:
df

,review,sentiment,Tokens
0,one reviewers mentioned watching 1 oz episode ...,positive,o n e r e v i e w e r s m e n t i o n e d ...
1,wonderful little production br br filming tech...,positive,w o n d e r f u l l i t t l e p r o d u c ...
2,thought wonderful way spend Tears Eyes hot sum...,positive,t h o u g h t w o n d e r f u l w a y s ...
3,basically theres family little boy jake thinks...,negative,b a s i c a l l y t h e r e s f a m i l ...
4,petter matteis love Tears Eyes money visually ...,positive,p e t t e r m a t t e i s l o v e T e a ...
...,...,...,...
49995,thought movie right good job wasnt creative or...,positive,t h o u g h t m o v i e r i g h t g o o ...
49996,bad plot bad dialogue bad acting idiotic direc...,negative,b a d p l o t b a d d i a l o g u e b ...
49997,catholic taught parochial elementary schools n...,negative,c a t h o l i c t a u g h t p a r o c h i ...
49998,im going disagree previous comment side maltin...,negative,i m g o i n g d i s a g r e e p r e v ...


In [ ]:
df['Tokens'] = df['Tokens'].apply(lambda tokens: ' '.join(tokens))  # Join tokens into a single string

In [ ]:
X=df['Tokens']
y=df['sentiment']


In [ ]:
X

,Tokens
0,o n e r e v i e w e r s m e n t i o n e d ...
1,w o n d e r f u l l i t t l e p r o d u c ...
2,t h o u g h t w o n d e r f u l w a y s ...
3,b a s i c a l l y t h e r e s f a m i l ...
4,p e t t e r m a t t e i s l o v e T e a ...
...,...
49995,t h o u g h t m o v i e r i g h t g o o ...
49996,b a d p l o t b a d d i a l o g u e b ...
49997,c a t h o l i c t a u g h t p a r o c h i ...
49998,i m g o i n g d i s a g r e e p r e v ...


In [ ]:
y = y.map({'positive':1,'negative':0})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.9, min_df=5)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [ ]:
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.50      1.00      0.66      4961
           1       0.00      0.00      0.00      5039

    accuracy                           0.50     10000
   macro avg       0.25      0.50      0.33     10000
weighted avg       0.25      0.50      0.33     10000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

NameError: name 'accuracy' is not defined

In [ ]:

new_review = "The ending was very good but the overall movie was not that good as it was quite boring"
new_review_tfidf = vectorizer.transform([new_review])
new_prediction = model.predict(new_review_tfidf)
new_prediction_label = 'positive' if new_prediction == 1 else 'negative'

print(f"Review: {new_review}")
print(f"Prediction: {new_prediction_label}")

Review: The ending was very good but the overall movie was not that good as it was quite boring
Prediction: negative


In [ ]:
df=pd.read_csv("clean.csv")

In [ ]:
df

,review,sentiment,Tokens
0,one reviewers mentioned watching 1 oz episode ...,positive,o n e r e v i e w e r s m e n t i o n e d ...
1,wonderful little production br br filming tech...,positive,w o n d e r f u l l i t t l e p r o d u c ...
2,thought wonderful way spend Tears Eyes hot sum...,positive,t h o u g h t w o n d e r f u l w a y s ...
3,basically theres family little boy jake thinks...,negative,b a s i c a l l y t h e r e s f a m i l ...
4,petter matteis love Tears Eyes money visually ...,positive,p e t t e r m a t t e i s l o v e T e a ...
...,...,...,...
49995,thought movie right good job wasnt creative or...,positive,t h o u g h t m o v i e r i g h t g o o ...
49996,bad plot bad dialogue bad acting idiotic direc...,negative,b a d p l o t b a d d i a l o g u e b ...
49997,catholic taught parochial elementary schools n...,negative,c a t h o l i c t a u g h t p a r o c h i ...
49998,im going disagree previous comment side maltin...,negative,i m g o i n g d i s a g r e e p r e v ...
